## Exercise 2

# Root finding
Program a method for root finding by bisection (interval halving)

Find the root of sin(x) between 3 and 4 by the bisection method

In [1]:
import numpy as np
a, b = 3, 4
def f(x): return np.sin(x)

In [2]:
def bisect(f, a, b, maxiter=53):
    # task: define the body of this function
    # check input and loop bisection until maxiter (or other stopping criterium) reached
    # return argmin(f) and number of iterations
    if np.sign(f(a))*np.sign(f(b)) >= 0:
        raise(ValueError("Function sign must differ at a and b"))
    
    for i in range(maxiter):
        m = (a+b)/2.
        fm = f(m)
        if m in [a, b] or fm == 0:
            # floating point tolerance reached or exact solution found
            return m, i
        
        if fm*np.sign(f(a)) < 0:
            b = m
        elif fm*np.sign(f(b)) < 0:
            a = m
    
    return m, i

In [3]:
bisect(f, 3, 4)

(3.141592653589793, 51)

In [4]:
np.pi

3.141592653589793

### Newton's method

Function $f(x)$ can be approximated by Taylor series in the vicinity of point $x_n$ as 
$$ f(x) \approx f(x_{n}) + f'(x_n)(x - x_n)$$
Solving for $f(x)=0$ gives
$$x = x_{n}-\frac{f(x_n)}{f'(x_n)}$$

In [5]:
# given
def f(x): return np.sin(x)
def df(x): return np.cos(x)

In [6]:
# implement the Newton's method
def newton(f, df, a, maxiter=10):
    for i in range(maxiter):
        a_new = a - f(a)/df(a)
        if a_new == a:
            return a, i
        a = a_new
    return a, i

In [7]:
newton(f, df, 4.)

(3.141592653589793, 4)

In [8]:
newton(f, df, 4.8) # the method may be unstable

(15.707963267948966, 5)

### Optimization

In [9]:
%%timeit
bisect(f, 3, 4)

81.6 µs ± 828 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [10]:
%%timeit
newton(f, df, 4.)

4.21 µs ± 56.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [11]:
from numba import njit

In [12]:
# implement the Newton's method
@njit
def jitnewton(f, df, a, maxiter=10):
    for i in range(maxiter):
        a_new = a - f(a)/df(a)
        if a_new == a:
            return a, i
        a = a_new
    return a, i
# given
@njit
def jitf(x): return np.sin(x)

@njit
def jitdf(x): return np.cos(x)


In [13]:
@njit
def jitbisect(f, a, b, maxiter=53):
    # task: define the body of this function
    if np.sign(f(a))*np.sign(f(b)) >= 0:
        raise(ValueError("Function sign must differ at a and b"))
    
    for i in range(maxiter):
        m = (a+b)/2.
        fm = f(m)
        if m in [a, b] or fm == 0:
            # floating point tolerance reached or exact solution found
            return m, i
        
        if fm*np.sign(f(a)) < 0:
            b = m
        elif fm*np.sign(f(b)) < 0:
            a = m
    
    return m, i

In [14]:
jitnewton(jitf, jitdf, 4.), jitbisect(jitf, 3, 4)

((3.141592653589793, 4), (3.141592653589793, 51))

In [15]:
%%timeit
jitbisect(jitf, 3., 4.)

21 µs ± 364 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [16]:
%%timeit
jitnewton(jitf, jitdf, 4.)

6.08 µs ± 55.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [17]:
@njit
def jitnewton_global(a, maxiter=10):
    for i in range(maxiter):
        a_new = a - jitf(a)/jitdf(a)
        if a_new == a:
            return a, i
        a = a_new
    return a, i

In [18]:
%%timeit
jitnewton_global(4.)

238 ns ± 1.59 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [19]:
# implement the Newton's method using closure
def jitnewton_generator(f, df, a, maxiter=10):
    @njit
    def jitnewton(a, maxiter=10):
        for i in range(maxiter):
            a_new = a - f(a)/df(a)
            if a_new == a:
                return a, i
            a = a_new
        return a, i
    return jitnewton

jitnewton_generated = jitnewton_generator(jitf, jitdf, a, maxiter=10)

In [20]:
%%timeit
jitnewton_generated(4.)

238 ns ± 2.37 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
